# Global Sensitivity B — `n_prior_periods` Audit

이 Notebook은 공식 Global Stage 1~4 결과를 바꾸지 않는다. 기존 고정 Global **Train** 행에서 Stage 3의 확정 25개 Feature에 `n_prior_periods`를 더한 26개 Feature 구조만 점검한다. 모델 학습·CV·OOF·Test 접근·예측·metric 계산은 포함하지 않는다.

`n_prior_periods` 값은 `build_person_period_dataset()`이 Global Person-Period 전체 이력에서 계산해 저장한 값을 재사용한다.

## 1. 입력 경로와 공통 함수

공식 baseline 저장 산출물, 고정 split, Stage 3 Feature 목록만 지정합니다. 이 단계에서는 데이터를 계산하거나 모델을 학습하지 않습니다.


In [2]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path(os.environ.get('KHUDA_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'code').is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError('KHUDA_PROJECT_ROOT에 저장소 루트를 지정하거나 저장소 안에서 Notebook을 실행하세요.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if 'code' in sys.modules and not hasattr(sys.modules['code'], '__path__'):
    del sys.modules['code']

from code.pipeline.audit import (
    attach_person_period_column,
    load_saved_global_train_frame,
    load_selected_feature_names,
)

RESULT_ROOT = ROOT / 'data' / 'result' / 'baseline_42features'
DATASET_PATH = RESULT_ROOT / 'datasets' / 'global_dataset.parquet'
PERSON_PERIOD_PATH = RESULT_ROOT / 'datasets' / 'person_period.parquet'
SPLIT_PATH = RESULT_ROOT / 'splits' / 'split_ids.csv'
SELECTED_FEATURES_PATH = RESULT_ROOT / 'modeling' / 'stage_3' / 'selected_features.csv'

required_paths = [DATASET_PATH, PERSON_PERIOD_PATH, SPLIT_PATH, SELECTED_FEATURES_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('필요한 공식 Global 저장 산출물이 없습니다:' + chr(10) + chr(10).join(missing_paths))

## 2. Global Train 및 Feature 계약

기존 Train SAMPID만 선택하고, `selected_features.csv`의 25개 Feature에 `n_prior_periods` 하나를 추가해 26개인지 확인합니다.


In [3]:
# 기존 Train SAMPID만 사용한다. Test Dataset은 만들지 않는다.
train_frame = load_saved_global_train_frame(DATASET_PATH, SPLIT_PATH)
selected_features = load_selected_feature_names(SELECTED_FEATURES_PATH)
train_with_prior = attach_person_period_column(
    train_frame, PERSON_PERIOD_PATH, 'n_prior_periods'
)

missing_features = [feature for feature in selected_features if feature not in train_with_prior.columns]
if missing_features:
    raise ValueError('저장된 Global Train에 Stage 3 선택 Feature가 없습니다: ' + ', '.join(missing_features))
B_features = selected_features + ['n_prior_periods']
X_B = train_with_prior.loc[:, B_features].copy()

display(pd.DataFrame([{
    'train_rows': len(train_with_prior),
    'train_unique_SAMPID': train_with_prior['SAMPID'].nunique(),
    'existing_selected_feature_count': len(selected_features),
    'B_feature_count': X_B.shape[1],
    'n_prior_periods_missing_count': train_with_prior['n_prior_periods'].isna().sum(),
    'test_used': False,
}]))
display(pd.DataFrame({'feature_order': range(1, len(B_features) + 1), 'feature': B_features}))

,train_rows,train_unique_SAMPID,existing_selected_feature_count,B_feature_count,n_prior_periods_missing_count,test_used
0,11925,5737,25,26,0,False


,feature_order,feature
0,1,gender
1,2,age
2,3,region_5
3,4,baseline_year
4,5,education_level
5,6,student_status
6,7,student_type
7,8,university_type
8,9,major_group
9,10,months_since_graduation


## 3. `n_prior_periods` 분포

전체 Train 분포와 baseline year별 분포를 표로 확인합니다.


In [4]:
# n_prior_periods 단변량 분포와 기준연도별 분포
display(train_with_prior['n_prior_periods'].value_counts(dropna=False).sort_index().rename('row_count').to_frame())
display(train_with_prior['n_prior_periods'].agg(['min', 'max', 'mean']).to_frame('n_prior_periods'))
baseline_year_distribution = (
    train_with_prior.groupby('baseline_year')['n_prior_periods']
    .agg(['count', 'min', 'max', 'mean'])
    .reset_index()
)
display(baseline_year_distribution)

,row_count
n_prior_periods,
0,5737
1,3842
2,2346


,n_prior_periods
min,0.000000
max,2.000000
mean,0.715639


,baseline_year,count,min,max,mean
0,2021,4809,0,0,0.000000
1,2022,3923,0,1,0.860821
2,2023,3193,0,2,1.615096


## 4. 사람별 반복관측과 연대기 예시

SAMPID별 Person-Period 행 수와 `n_prior_periods`의 관계를 확인하고, 반복관측이 많은 일부 SAMPID의 연대기를 직접 검토합니다.


In [5]:
# 사람별 반복 행 수와 n_prior_periods의 관계
person_period_relationship = (
    train_with_prior.groupby('SAMPID')['n_prior_periods']
    .agg(person_period_rows='size', n_prior_min='min', n_prior_max='max', n_prior_mean='mean')
    .reset_index()
    .sort_values(['person_period_rows', 'SAMPID'], ascending=[False, True])
)
display(person_period_relationship)

# 반복관측 수가 많은 사람부터 고정된 수의 사례를 골라 연도 순서를 사람이 직접 확인한다.
example_sampids = person_period_relationship['SAMPID'].head(5)
chronology_examples = (
    train_with_prior.loc[train_with_prior['SAMPID'].isin(example_sampids), [
        'SAMPID', 'baseline_year', 'target_year', 'n_prior_periods'
    ]]
    .sort_values(['SAMPID', 'baseline_year', 'target_year'])
    .reset_index(drop=True)
)
display(chronology_examples)

,SAMPID,person_period_rows,n_prior_min,n_prior_max,n_prior_mean
0,100002,3,0,2,1.0
1,100003,3,0,2,1.0
3,100007,3,0,2,1.0
4,100008,3,0,2,1.0
6,100015,3,0,2,1.0
...,...,...,...,...,...
5722,112178,1,0,0,0.0
5724,112181,1,0,0,0.0
5728,112198,1,0,0,0.0
5732,112203,1,0,0,0.0


,SAMPID,baseline_year,target_year,n_prior_periods
0,100002,2021,2022,0
1,100002,2022,2023,1
2,100002,2023,2024,2
3,100003,2021,2022,0
4,100003,2022,2023,1
5,100003,2023,2024,2
6,100007,2021,2022,0
7,100007,2022,2023,1
8,100007,2023,2024,2
9,100008,2021,2022,0


## 5. 실행 시 검증할 assertions

아래 assertion은 사람이 Notebook을 실행할 때만 동작합니다. 미래 관측이 사용되지 않았는지와 Feature 수를 자동 점검합니다.


In [6]:
# Audit assertions — 이 셀은 사람이 Notebook을 실행할 때만 검사한다.
chronological = train_with_prior.sort_values(['SAMPID', 'baseline_year', 'target_year']).copy()
expected_prior_count = chronological.groupby('SAMPID').cumcount()
first_eligible_rows = chronological.groupby('SAMPID', sort=False).head(1)

assert len(selected_features) == 25, 'Stage 3 selected_features.csv는 정확히 25개여야 합니다.'
assert X_B.shape[1] == 26, 'B Feature 수는 26개여야 합니다.'
assert chronological['n_prior_periods'].ge(0).all(), 'n_prior_periods는 0 이상이어야 합니다.'
assert first_eligible_rows['n_prior_periods'].eq(0).all(), '각 SAMPID의 첫 eligible 행은 0이어야 합니다.'
assert chronological.groupby('SAMPID')['baseline_year'].diff().fillna(0).ge(0).all(), '동일 SAMPID 안에서 baseline_year chronology가 역전되었습니다.'
assert chronological['target_year'].eq(chronological['baseline_year'] + 1).all(), 'target_year는 baseline_year 다음 해여야 합니다.'
assert chronological['n_prior_periods'].reset_index(drop=True).eq(expected_prior_count.reset_index(drop=True)).all(), (
    'n_prior_periods가 현재 행 이전의 eligible 관측 수와 일치하지 않습니다. 미래 관측 사용 여부를 확인하세요.'
)
print('All B audit assertions passed.')

All B audit assertions passed.
